# 05 - Adaptive

Plan section 39: the signal-informed controller mutates a live model under matched budgets, with comparable per-arm reports.

Reproducibility contract (plan section 40): every session begins by
determining its configuration, locating and validating prior state,
and continues from there; it ends by saving metrics and a summary JSON.
Run inside a Kaggle session (or locally) after checking out this
repository, with the repository root as the working directory.


In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd()
assert (REPO / 'src' / 'act').is_dir(), (
    f'run this notebook with the repository root as the working directory; '
    f'cwd was {REPO!r}'
)
sys.path.insert(0, str(REPO / 'src'))
print('python', sys.version.split()[0], '| repo', REPO)


# Adaptive topology evolution (plan section 39)

The actual topology-evolving model: three arms in one matched-
budget experiment - the fixed control, matched random churn, and
the adaptive controller deciding mutations from measurable
signals. The comparable per-arm ablation report is the notebook's
scientific output.


In [ ]:
from act.evolution.experiment import ExperimentConfig, run_experiment

result = run_experiment(ExperimentConfig(
    seed=42, steps=48, arms='fixed,random,adaptive', device='cpu',
))
arms = {arm.mode: arm for arm in result.arms}
assert arms['fixed'].mutation_count == 0
assert arms['random'].mutation_count == arms['adaptive'].mutation_count
for arm in result.arms:
    print(f"{arm.mode:>8}: loss={arm.validation_loss:.4f} "
          f"mutations={arm.mutation_count} params={arm.parameter_count}")

In [ ]:
# The adaptive arm's mutation history names the policy behind each
# structural event, so every decision is inspectable afterwards.
adaptive = arms['adaptive']
assert all(event['policy'] for event in adaptive.mutation_history)
for event in adaptive.mutation_history:
    print(event['step'], event['policy'], event['operation'])

In [ ]:
# Comparable reports: efficiency against total FLOPs and ablation
# deltas against the fixed control.
from act.evaluation.reports import ablation_report, efficiency_report

efficiency = efficiency_report(result)
ablation = ablation_report(result, reference_mode='fixed')
assert {row['mode'] for row in efficiency['rows']} == {'fixed', 'random', 'adaptive'}
assert {row['mode'] for row in ablation['comparisons']} == {'random', 'adaptive'}
print('adaptive - random:', round(efficiency['adaptive_minus_random'], 6))
for row in ablation['comparisons']:
    print(row['mode'], 'delta:', round(row['validation_loss_delta'], 6))

In [ ]:
import json
from pathlib import Path

record_dir = Path('artifacts/notebook-records')
record_dir.mkdir(parents=True, exist_ok=True)
payload = {'notebook': '05_adaptive',
 'adaptive_minus_random': efficiency['adaptive_minus_random'],
 'rows': efficiency['rows'],
 'comparisons': ablation['comparisons']}
record = json.dumps(payload, sort_keys=True, indent=2) + '\n'
target = record_dir / '05-adaptive.json'
existing = target.read_text() if target.exists() else None
if existing != record:
    target.write_text(record)
print('record:', target)